In [1]:
import clickhouse_connect

client = clickhouse_connect.get_client(
    host='localhost', port=8123,
    username='default', password='homework'
)

Создадим базу данных student1.

In [10]:
client.command("""
CREATE DATABASE IF NOT EXISTS student1 ON CLUSTER homework_cluster
""")
print("Создана база данных student1")

Создана база данных student1


### Задание 1
Создадим реплицируемую таблицу **push_events_raw_local** и скопируем данные из базы данных.

In [ ]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_raw_local ON CLUSTER homework_cluster
(
event_dt        DateTime,
event_id        UInt64,
user_id         UInt64,
message_id      UInt64,
platform        LowCardinality(String),
status          LowCardinality(String),
latency_ms      UInt32
)
ENGINE = ReplicatedMergeTree(
    '/clickhouse/tables/{shard}/student1/push_events_raw_local',
    '{replica}'
)
PARTITION BY toYYYYMMDD(event_dt)   -- партицируем по дню. Так удобнее для TTl.
ORDER BY (user_id, event_dt)
TTL event_dt + INTERVAL 30 DAY      -- строки старше 30 дней удалются.
""")
print("Создана таблица push_events_raw_local")

Создана таблица push_events_raw_local


Создадим распределенную таблицу.

In [9]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_raw ON CLUSTER homework_cluster
AS student1.push_events_raw_local
ENGINE = Distributed(
    'homework_cluster',          -- кластер (по нодам какого кластера нам проходиться)
    'student1',                  -- база данных (в какой базе искать на этих нодах )
    'push_events_raw_local',     -- локальная таблица
    rand()                       -- равномерное распределение
)
""")
print("Создана распределенная таблица push_events_raw.")

Создана распределенная таблица push_events_raw.


Добавим данные в таблицы.

In [ ]:
client.command("""
INSERT INTO student1.push_events_raw
SELECT
    event_dt,
    event_id,
    user_id,
    message_id,
    platform,
    status,
    latency_ms
FROM homework_data.push_events_source
""")
print("Данные вставлены")

Данные вставлены


Проверка:

In [ ]:
print(
     client.query_df("""
                    SELECT 
                        hostName() AS host,
                        count() AS rows
                    FROM student1.push_events_raw
                    GROUP BY host
                    """).to_string(index=False)
     )

host    rows
ch11 1046730
ch22 1048342


In [13]:
print(
    client.query_df("""
                    SELECT 
                        hostName() AS host,
                        database, table,
                        is_readonly,
                        absolute_delay,
                        log_pointer
                    FROM clusterAllReplicas('homework_cluster', system.replicas)
                    WHERE database = 'student1'
                    """).to_string(index=False))

host database                 table  is_readonly  absolute_delay  log_pointer
ch11 student1 push_events_raw_local            0               0           33
ch22 student1 push_events_raw_local            0               0           33
ch21 student1 push_events_raw_local            0               0           33
ch12 student1 push_events_raw_local            0               0           33


Ответ на вопрос:
если вставлеять данные в локальную таблицу, то данные попадут только в один шард (который лежит на ноде ch11) и в его реплику (которая лежит на ноде sh12).

### Задание 2.

Создадим агрегирующую таблицу.

Таблица никаких аргегаций не делате, она лишь хранит наши данные. Агрегирующие таблицы хранят не отдельные строки, а типо их совокупность.

In [3]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_agg_local ON CLUSTER homework_cluster
(
    platform              LowCardinality(String),
    event_date            Date,
    events_count          AggregateFunction(count),
    uniq_users_id         AggregateFunction(uniq, UInt64),  -- в аргументах сперва идет тип операции, а замет тип входных данных.
    avg_latency_ms        AggregateFunction(avg, UInt32)
)
ENGINE = ReplicatedAggregatingMergeTree(
    '/clickhouse/tables/{shard}/student1/push_events_agg_local',
    '{replica}'
)
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform) 
""")
print("push_events_agg_local создана")

push_events_agg_local создана


Взял ReplicatedAggregatingMergeTree вместо AggregatingMergeTree, поскольку MV срабатывает только на реплике, куда пришла вставка, а репликация raw-таблицы MV на второй реплике не запускает. Без Replicated вторая реплика каждого шарда осталась бы без агрегатов, и запросы через Distributed давали бы разные результаты в зависимости от выбранной реплики.

Добавим распределенную таблицу.

In [4]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_agg ON CLUSTER homework_cluster
AS student1.push_events_agg_local
ENGINE = Distributed(
    'homework_cluster',
    'student1',
    'push_events_agg_local',
    rand()
)
""")
print("Создана распределенная таблица push_events_agg.")

Создана распределенная таблица push_events_agg.


Создадим MATERIALIZED VIEW.

In [ ]:
client.command("""
CREATE MATERIALIZED VIEW IF NOT EXISTS student1.push_events_agg_mv ON CLUSTER homework_cluster
TO student1.push_events_agg_local
AS SELECT
    toDate(event_dt)          AS event_date,     -- из DateTime делаем Date
    platform,
    countState()              AS events_count,
    uniqState(user_id)        AS uniq_users_id,  -- по колонке user_id считаем кол-во уникальных пользователей.
    avgState(latency_ms)      AS avg_latency_ms  -- по колонке latency_ms учитывается среднее.
FROM student1.push_events_raw_local
GROUP BY event_date, platform
""")
print("MV push_events_agg_mv создан")

MV push_events_agg_mv создан


Проверки:
Для проверки очистим push_events_raw и добавим снова данные из базы данных, чтобы их заметил агрегатор.

In [6]:
# Очистили таблицу.
client.command("""
TRUNCATE TABLE student1.push_events_raw_local 
ON CLUSTER homework_cluster
""")
print(
    "Строк в raw после очистки:",
    client.query(
                "SELECT count() FROM student1.push_events_raw"
                ).first_row[0]
    )

# Добавим данные снова.
client.command("""
INSERT INTO student1.push_events_raw
SELECT
    event_dt,
    event_id,
    user_id,
    message_id,
    platform,
    status,
    latency_ms
FROM homework_data.push_events_source
""")
print("Данные вставлены")

Строк в raw после очистки: 0
Данные вставлены


Сделаем запрос для проверки работоспособности агрегатора. 

In [ ]:
df_agg = client.query_df("""
SELECT
    event_date,
    platform,
    countMerge(events_count)              AS events,
    uniqMerge(uniq_users_id)              AS unique_users,
    round(avgMerge(avg_latency_ms), 2)    AS avg_latency
FROM student1.push_events_agg
GROUP BY event_date, platform
ORDER BY event_date, platform
""")
print(df_agg.to_string(index=False))

event_date platform  events  unique_users  avg_latency
2026-09-08  ANDROID   11126          6508      2510.74
2026-09-08  DESKTOP   11099          6583      2501.99
2026-09-08      IOS   11100          6505      2521.67
2026-09-08      WEB   10952          6479      2474.72
2026-09-09  ANDROID   17551          8080      2523.30
2026-09-09  DESKTOP   17502          8164      2492.88
2026-09-09      IOS   17428          8098      2504.90
2026-09-09      WEB   17528          8100      2504.18
2026-09-10  ANDROID   17652          8211      2504.35
2026-09-10  DESKTOP   17462          8103      2497.11
2026-09-10      IOS   17435          8129      2511.81
2026-09-10      WEB   17343          8082      2507.82
2026-09-11  ANDROID   17459          8139      2489.68
2026-09-11  DESKTOP   17755          8161      2496.74
2026-09-11      IOS   17392          8109      2489.76
2026-09-11      WEB   17621          8178      2501.98
2026-09-12  ANDROID   17502          8075      2513.89
2026-09-12

Проверим сколько в таблице было записей до агрегации.

In [ ]:
print(
    client.query(
    """
    SELECT count()
    FROM student1.push_events_agg
    """
    ).first_row[0])

248


In [ ]:
client.command("""
    OPTIMIZE TABLE student1.push_events_agg_local  -- объединили парты на нодах.
    ON CLUSTER homework_cluster FINAL
    """)

print(
    client.query(
    """
    SELECT count()
    FROM student1.push_events_agg
    """
    ).first_row[0])

248


Запрос в таблицу raw.

In [12]:
df_raw = client.query_df("""
SELECT
    toDate(event_dt)             AS event_date,
    platform,
    count()                      AS events,
    uniq(user_id)                AS unique_users,
    round(avg(latency_ms), 2)    AS avg_latency
FROM student1.push_events_raw
GROUP BY event_date, platform
ORDER BY event_date, platform
""")
df_raw.head()

,event_date,platform,events,unique_users,avg_latency
0,2026-09-08,ANDROID,11126,6508,2510.74
1,2026-09-08,DESKTOP,11099,6583,2501.99
2,2026-09-08,IOS,11100,6505,2521.67
3,2026-09-08,WEB,10952,6479,2474.72
4,2026-09-09,ANDROID,17551,8080,2523.30


In [ ]:
df_agg = client.query_df("""
SELECT
    event_date,
    platform,
    countMerge(events_count)              AS events,
    uniqMerge(uniq_users_id)              AS unique_users,
    round(avgMerge(avg_latency_ms), 2)    AS avg_latency
FROM student1.push_events_agg
GROUP BY event_date, platform
ORDER BY event_date, platform
""")
df_agg.head()

,event_date,platform,events,unique_users,avg_latency
0,2026-09-08,ANDROID,11126,6508,2510.74
1,2026-09-08,DESKTOP,11099,6583,2501.99
2,2026-09-08,IOS,11100,6505,2521.67
3,2026-09-08,WEB,10952,6479,2474.72
4,2026-09-09,ANDROID,17551,8080,2523.30


Результаты запросов совпадают.

Если создать MV после вставки в raw, то он будет пустым, посколько MV отслеживает вставки после своего создания.

Если не будет Group by, то результат будет корректным, но посчитан будет по всей raw-таблице, а мы хоитм чтобы считалось среднее по группам в зависимости от даты события и платформы.

### Задание 3

Создадим таблицу push_counters_local.

In [14]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_counters_local ON CLUSTER homework_cluster
(
    platform              LowCardinality(String),
    event_date            Date,
    delivered_count       UInt64,
    failed_count          UInt64,
    opened_count          UInt64,
    user_id               UInt64
)
ENGINE = ReplicatedSummingMergeTree(
    '/clickhouse/tables/{shard}/student1/push_counters_local',
    '{replica}'
)
PARTITION BY toYYYYMM(event_date)
ORDER BY (event_date, platform) 
""")
print("push_counters_local создана")

push_counters_local создана


Выбран ReplicatedSummingMergeTree по той же самой причине, что и в прошлом задании.

Создадим распределенную таблицу.

In [15]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_counters ON CLUSTER homework_cluster
AS student1.push_counters_local
ENGINE = Distributed(
    'homework_cluster',  
    'student1',               
    'push_counters_local',     
    rand()               
    )
""")
print("Создана распределенная таблица push_counters.")

Создана распределенная таблица push_counters.


Создадим MV.

In [16]:
client.command("""
CREATE MATERIALIZED VIEW IF NOT EXISTS student1.push_counters_mv ON CLUSTER homework_cluster
TO student1.push_counters_local
AS SELECT
    platform,
    toDate(event_dt) as event_date,
    countIf(status = 'delivered') as delivered_count,
    countIf(status = 'failed') AS failed_count,
    countIf(status = 'opened') AS opened_count,
    user_id
FROM student1.push_events_raw_local
GROUP BY (event_date, platform, user_id)
""")
print("MV push_counters_mv создан")

MV push_counters_mv создан


Проверим кол-во строк до добавления их в таблицу push_events_raw_local.

In [ ]:
print(client.query_df("""
SELECT
    count()               AS rows,      -- кол-во строчек.
    sum(delivered_count)  AS delivered, -- сумма по всем строчкам.
    sum(failed_count)     AS failed,
    sum(opened_count)     AS opened
FROM student1.push_counters
""").to_string(index=False))

 rows  delivered  failed  opened
    0          0       0       0


Остановим слияние счетчиков чтобы ответить на вопрос, что произойдет после OPTIMIZE FINAL.

In [18]:
client.command("SYSTEM STOP MERGES ON CLUSTER homework_cluster student1.push_counters_local")


['ch21',
 '9000',
 '0',
 '',
 '3',
 '0\nch22',
 '9000',
 '0',
 '',
 '2',
 '0\nch12',
 '9000',
 '0',
 '',
 '1',
 '0\nch11',
 '9000',
 '0',
 '',
 '0',
 '0']

Очистим raw и агрегаты чтобы не было повторения строк и зальем их повторно.

In [19]:
client.command("TRUNCATE TABLE student1.push_events_raw_local ON CLUSTER homework_cluster")
client.command("TRUNCATE TABLE student1.push_events_agg_local ON CLUSTER homework_cluster")

client.command("""
INSERT INTO student1.push_events_raw
SELECT
    event_dt,
    event_id,
    user_id,
    message_id,
    platform,
    status,
    latency_ms
FROM homework_data.push_events_source
""")
print("Данные вставлены")

Данные вставлены


Проверим состояние счетчиков.

In [20]:
print(client.query_df("""
SELECT
    count()               AS rows,      -- кол-во строчек.
    sum(delivered_count)  AS delivered, -- сумма по всем строчкам.
    sum(failed_count)     AS failed,
    sum(opened_count)     AS opened
FROM student1.push_counters
""").to_string(index=False))

 rows  delivered  failed  opened
  248     699292  700082  700162


Посмотрим на user_id.

In [21]:
print(client.query_df("""
SELECT * FROM student1.push_counters
ORDER BY event_date, platform
LIMIT 10
""").to_string(index=False))

platform event_date  delivered_count  failed_count  opened_count  user_id
 ANDROID 2026-09-08             1899          1778          1873 20935248
 ANDROID 2026-09-08             1879          1820          1877 20960472
 DESKTOP 2026-09-08             1904          1922          1846 21704084
 DESKTOP 2026-09-08             1787          1777          1863 20727750
     IOS 2026-09-08             1852          1773          1856 20509479
     IOS 2026-09-08             1834          1850          1935 21011093
     WEB 2026-09-08             1843          1821          1755 20528131
     WEB 2026-09-08             1891          1806          1836 20903940
 ANDROID 2026-09-09             2924          2970          2914 28624878
 ANDROID 2026-09-09             2900          2893          2950 28708558


In [38]:
client.command("SYSTEM START MERGES ON CLUSTER homework_cluster student1.push_counters_local")

['ch22',
 '9000',
 '0',
 '',
 '3',
 '0\nch12',
 '9000',
 '0',
 '',
 '2',
 '0\nch21',
 '9000',
 '0',
 '',
 '1',
 '0\nch11',
 '9000',
 '0',
 '',
 '0',
 '0']

In [39]:
client.command("OPTIMIZE TABLE student1.push_counters_local ON CLUSTER homework_cluster FINAL")

print(client.query_df("""
SELECT * FROM student1.push_counters
ORDER BY event_date, platform
LIMIT 10
""").to_string(index=False))

platform event_date  delivered_count  failed_count  opened_count  user_id
 ANDROID 2026-09-08             1899          1778          1873 20935248
 ANDROID 2026-09-08             1879          1820          1877 20960472
 DESKTOP 2026-09-08             1904          1922          1846 21704084
 DESKTOP 2026-09-08             1787          1777          1863 20727750
     IOS 2026-09-08             1852          1773          1856 20509479
     IOS 2026-09-08             1834          1850          1935 21011093
     WEB 2026-09-08             1843          1821          1755 20528131
     WEB 2026-09-08             1891          1806          1836 20903940
 ANDROID 2026-09-09             2924          2970          2914 28624878
 ANDROID 2026-09-09             2900          2893          2950 28708558


user_id до и после OPTIMIZE TABLE одинаковые, поскольку SummingMergeTree сложил объекты с одинаковыми ключами группировки в один и при этом все числовые столбцы сложил, которые не являлись ключами. Поэтому такой большой user_id. А не помог OPTIMIZE TABLE поскольку слились данные ещё при вставке.

Это можно исправить, если убрать user_id из таблицы и из MV, чтобы он его не суммировал, либо сделать его ключем группировки.

Двигатель SummingMergeTree не подходит для среднего, поскольку он только складывает, а сложением среднее значение не получить.

### Задача 4

Создадим таблицу push_events_dedup_local

In [ ]:
# Нету партиций, поскольку поскольку дубли будут удаляться если они лежат в одной партиции.
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_dedup_local ON CLUSTER homework_cluster
(
    event_dt        DateTime,
    event_id        UInt64,
    user_id         UInt64,
    message_id      UInt64,
    platform        LowCardinality(String),
    status          LowCardinality(String),
    latency_ms      UInt32
)
ENGINE = ReplicatedReplacingMergeTree(
    '/clickhouse/tables/{shard}/student1/push_events_dedup_local',
    '{replica}',
    event_dt                     -- при схлопывании остаётся строка с максимальным event_dt.
)
ORDER BY event_id                -- по этой колонке считаем что дубль.
""")
print("push_events_dedup_local создана")

push_events_dedup_local создана


Создадим распределенную таблицу push_events_dedup

In [41]:
client.command("""
CREATE TABLE IF NOT EXISTS student1.push_events_dedup ON CLUSTER homework_cluster
AS student1.push_events_dedup_local
ENGINE = Distributed(
    'homework_cluster',
    'student1',
    'push_events_dedup_local',
    cityHash64(event_id)         -- взял такое шардирование, чтобы с одним и тем же ключом строки попадали в одни и те же шарды.
)
""")
print("push_events_dedup создана")

push_events_dedup создана


Вставим данные новую таблицу push_events_dedup.

In [ ]:
client.command("""
INSERT INTO student1.push_events_dedup
SELECT event_dt, event_id, user_id, message_id, platform, status, latency_ms
FROM homework_data.push_events_source
SETTINGS optimize_on_insert = 0         -- эта настройка позволяет дублям не пропадать.
""")
print("Данные вставлены")

Данные вставлены


Проверки:

In [44]:
print(client.query("SELECT count() FROM student1.push_events_dedup").first_row[0])
print(client.query("SELECT count() FROM student1.push_events_dedup FINAL").first_row[0])
print("уникальных event_id :", client.query("SELECT uniqExact(event_id) FROM homework_data.push_events_source").first_row[0])

2099536
2000000
уникальных event_id : 2000000


Результаты разные, потому что FINAL убрал часть дублей. FINAL при вызове удаляет дубли. Дубли удаляются при слиянии партов, оно происходит параллельным процессом.

Выиграет макисмальная версия, поскольку она будет новее чем прошлая.

Нельзя полагатьс на ReplacingMergeTree для строгой дедубликации, потому что она происходит фоном при слиянии партов, но не известно когда она пройдет. Дубли уничтожаются в одном шарде и одной партиции.

### Задание 5

Напишем запросы и сверим их время выполнения.

In [45]:
df_raw = client.query_df("""
SELECT
    platform,
    count()                     AS events,
    uniq(user_id)               AS unique_users,
    round(avg(latency_ms), 2)   AS avg_latency
FROM student1.push_events_raw
WHERE event_dt >= today() - 7
GROUP BY platform
ORDER BY platform
SETTINGS log_comment = 'hw5_raw'
""")
print(df_raw.to_string(index=False))

platform  events  unique_users  avg_latency
 ANDROID  129127         10000      2498.17
 DESKTOP  129222         10000      2493.18
     IOS  128137         10000      2499.92
     WEB  128455         10000      2497.80


In [48]:
df_agg = client.query_df("""
SELECT
    platform,
    countMerge(events_count)              AS events,
    uniqMerge(uniq_users_id)              AS unique_users,
    round(avgMerge(avg_latency_ms), 2)    AS avg_latency
FROM student1.push_events_agg
WHERE event_date >= today() - 7
GROUP BY platform
ORDER BY platform
SETTINGS log_comment = 'hw5_agg'
""")
print(df_agg.to_string(index=False))

platform  events  unique_users  avg_latency
 ANDROID  129127         10000      2498.17
 DESKTOP  129222         10000      2493.18
     IOS  128137         10000      2499.92
     WEB  128455         10000      2497.80


In [ ]:
client.command("SYSTEM FLUSH LOGS")

print(client.query_df("""
SELECT
    log_comment,      -- нужен чтобы различать запросы.
    query_duration_ms,
    read_rows,
    formatReadableSize(read_bytes) AS read_size
FROM system.query_log
WHERE type = 'QueryFinish'
  AND is_initial_query
  AND log_comment IN ('hw5_raw', 'hw5_agg')
  AND event_time > now() - INTERVAL 10 MINUTE
ORDER BY event_time DESC
""").to_string(index=False))

log_comment  query_duration_ms  read_rows read_size
    hw5_agg                 27         64  8.30 KiB
    hw5_raw                 43     514941  8.35 MiB


agg быстрее поскольку agg содержит в себе агрегаты посчитанные заранее из-за чего они занимает меньше строк. А при подсчете статистик через raw таблицу для этого надо пройтись по большему кол-ву строк и посчитать статистики.

raw может быть быстрее, если мало данных находится в raw. Также может быть, что агрегаты не схлопнулись и их довольно много. Ключ агрегатов слишком сложный.